# Eliminating Mathematical Hallucinations with Deterministic Tool Use

LLMs predict tokens — they don't compute. When asked "What is 347 × 893?", the model guesses which digits are most likely, not which are mathematically correct. This guide shows how to eliminate mathematical hallucinations entirely by routing computation to a deterministic engine.

## Key Finding

We benchmarked 94 math problems across models of different sizes:

| System | Accuracy | Speed | Size |
|--------|----------|-------|------|
| Small model (3B) | 55% | 200ms | 1.8 GB |
| Medium model (7B) | 77% | 300ms | 4.4 GB |
| Large model (32B) | 93% | 2,600ms | 18.5 GB |
| **SymPy Tool** | **100%** | **1.9ms** | **0 GPU** |

Scaling parameters from 3B to 32B (10x) only moves accuracy from 55% to 93%. It never reaches 100% because the failure is **architectural** — you can't guarantee mathematical correctness by sampling from a probability distribution. Tool use solves this.

## The Pattern: Separate Perception from Execution

1. The LLM handles **perception** — understanding what's being asked
2. A deterministic tool handles **execution** — computing the answer
3. The LLM handles **presentation** — formatting the result

This is the same principle behind code interpreter, but applied specifically to mathematical computation.

In [ ]:
# Install dependencies
!pip install openai sympy -q

In [ ]:
import json
import sympy
from openai import OpenAI

client = OpenAI()

# Define the math computation tool
math_tool = {
    "type": "function",
    "function": {
        "name": "compute_math",
        "description": "Compute an exact mathematical result using SymPy symbolic math. Use this for ANY calculation — arithmetic, algebra, trigonometry, statistics, finance, etc. Never compute math yourself; always use this tool.",
        "parameters": {
            "type": "object",
            "properties": {
                "expression": {
                    "type": "string",
                    "description": "A valid SymPy expression to evaluate. Examples: '347 * 893', 'sin(pi/6)', 'factorial(10)', '450000 * (0.065/12 * (1+0.065/12)**360) / ((1+0.065/12)**360 - 1)'"
                }
            },
            "required": ["expression"]
        }
    }
}

def compute_math(expression: str) -> dict:
    """Evaluate a math expression using SymPy. Zero hallucination."""
    try:
        # SymPy namespace for evaluation
        ns = {
            'sin': sympy.sin, 'cos': sympy.cos, 'tan': sympy.tan,
            'asin': sympy.asin, 'acos': sympy.acos, 'atan': sympy.atan,
            'sqrt': sympy.sqrt, 'log': sympy.log, 'exp': sympy.exp,
            'pi': sympy.pi, 'e': sympy.E,
            'factorial': sympy.factorial, 'binomial': sympy.binomial,
            'gcd': sympy.gcd, 'lcm': sympy.lcm,
            'Rational': sympy.Rational, 'N': sympy.N,
        }
        result = sympy.sympify(expression, locals=ns)
        numeric = float(sympy.N(result))
        return {"result": numeric, "exact": str(result), "verified": True}
    except Exception as e:
        return {"error": str(e), "verified": False}

## Demo: Problems LLMs Get Wrong

Let's test problems that LLMs commonly hallucinate on.

In [ ]:
test_problems = [
    ("What is 347 * 893?", 309871),
    ("What is the monthly payment on a $450,000 mortgage at 6.5% for 30 years?", 2844.31),
    ("What is 2 + 3 * 4?", 14),  # Order of operations
    ("What is 0.1 + 0.2?", 0.3),  # Float trap
    ("What is 15 factorial?", 1307674368000),
    ("What is the sine of 30 degrees?", 0.5),
]

# First: ask the model WITHOUT tools (raw token prediction)
print("=" * 60)
print("WITHOUT TOOLS (token prediction)")
print("=" * 60)

for problem, expected in test_problems:
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": f"{problem} Answer with ONLY the number."}],
        temperature=0,
    )
    answer = response.choices[0].message.content.strip()
    print(f"  {problem[:45]:45s} => {answer:>20s} (expected: {expected})")

In [ ]:
# Now: ask the model WITH the math tool
print("=" * 60)
print("WITH SYMPY TOOL (deterministic computation)")
print("=" * 60)

for problem, expected in test_problems:
    messages = [{"role": "user", "content": problem}]
    
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        tools=[math_tool],
        tool_choice="auto",
        temperature=0,
    )
    
    msg = response.choices[0].message
    
    if msg.tool_calls:
        # Model chose to use the tool — execute it
        call = msg.tool_calls[0]
        args = json.loads(call.function.arguments)
        result = compute_math(args["expression"])
        
        answer = result.get("result", "ERROR")
        tol = max(abs(expected) * 0.001, 0.01) if expected != 0 else 0.01
        correct = isinstance(answer, (int, float)) and abs(answer - expected) <= tol
        status = "CORRECT" if correct else "WRONG"
        print(f"  {problem[:45]:45s} => {str(answer):>20s} [{status}]")
    else:
        print(f"  {problem[:45]:45s} => {msg.content[:20]:>20s} [NO TOOL USED]")

## Why This Works

Token prediction **cannot guarantee mathematical correctness**. The model samples from a probability distribution over possible next tokens. For `347 × 893`, the correct answer (`309871`) and a plausible-looking wrong answer (`309,650`) have similar token probabilities.

SymPy performs **symbolic computation** — the same math engine behind tools like Wolfram Alpha. Given the same input, it always produces the same output. There's no randomness, no approximation, no hallucination.

The model's job becomes **perception** (understanding the question) and **presentation** (formatting the answer), not computation. This separation of concerns eliminates an entire class of failure.

## Scaling This Pattern

The [Math Swarm](https://github.com/michaelwinczuk/math-swarm) project extends this pattern to 12 categories (arithmetic, finance, trigonometry, combinatorics, statistics, healthcare) with 1,079 verified test problems at 100% accuracy, including 15 clinical healthcare formulas with guideline-based decision support.

The same principle applies beyond math:
- **Facts** → Knowledge graph lookup instead of memorized recall
- **Code** → Execute and verify instead of predict
- **Dates** → `datetime` library instead of token prediction

Anywhere an LLM computes, a deterministic tool can replace the computation with a guarantee.